In [ ]:
"""
Computational Electrocatalysis: Water Splitting Project (without plots)
"""

import numpy as np
import ipywidgets as widgets
from IPython.display import display, clear_output
from pyscf import gto, scf, mp, cc, dft
import contextlib
import io
import logging

try:
    from pyscf.geomopt.geometric_solver import optimize
except ImportError:
    print("Warning: geomeTRIC not found. Geometry optimization will be skipped.")
    optimize = lambda mf: mf.mol # Fallback if geomeTRIC is missing

def optimize_quietly(mf):
    """Runs PySCF geometry optimization while completely suppressing geomeTRIC logs."""
    f = io.StringIO()
    # Redirect both stdout and stderr
    with contextlib.redirect_stdout(f), contextlib.redirect_stderr(f):
        # Mute geomeTRIC's internal logger
        logger = logging.getLogger()
        old_level = logger.level
        logger.setLevel(logging.CRITICAL)
        try:
            opt_mol = optimize(mf)
        finally:
            # Restore the logger level once finished
            logger.setLevel(old_level)
    return opt_mol


# --- 1. Helper Functions for Chemistry & Geometry ---
BOHR_TO_ANGSTROM = 0.529177210903

def build_mol(coords, basis, spin=0):
    mol = gto.M(atom=coords, basis=basis, spin=spin, symmetry=False, verbose=0)
    return mol

def get_bond_length(mol, idx1, idx2):
    # PySCF returns coordinates in Bohr
    coords = mol.atom_coords() * BOHR_TO_ANGSTROM
    return np.linalg.norm(coords[idx1] - coords[idx2])

def get_bond_angle(mol, idx_a, idx_vertex, idx_c):
    coords = mol.atom_coords() * BOHR_TO_ANGSTROM
    v1 = coords[idx_a] - coords[idx_vertex]
    v2 = coords[idx_c] - coords[idx_vertex]
    cosine_angle = np.dot(v1, v2) / (np.linalg.norm(v1) * np.linalg.norm(v2))
    return np.degrees(np.arccos(np.clip(cosine_angle, -1.0, 1.0)))

def run_calculation(mol, method="HF", xc="pbe"):
    if method == "HF":
        mf = scf.UHF(mol) if mol.spin > 0 else scf.RHF(mol)
    elif method == "DFT":
        mf = dft.UKS(mol) if mol.spin > 0 else dft.RKS(mol)
        mf.xc = xc
    mf.kernel()
    return mf

# --- 2. Widget Definitions & UI Layout ---
out = widgets.Output()

# Global Inputs
basis_dropdown = widgets.Dropdown(options=['sto-3g', '3-21g', '6-31g', 'cc-pvdz', 'cc-pvtz'], value='sto-3g', description='Base Basis:')
h2_coords = widgets.Text(value='H 0 0 0; H 0 0 0.74', description='H2 Coords:')
o2_coords = widgets.Text(value='O 0 0 0; O 0 0 1.21', description='O2 Coords:')
h2o_coords = widgets.Text(value='O 0 0 0; H 0 0.757 0.586; H 0 -0.757 0.586', description='H2O Coords:')

# Task specific triggers
btn_task345 = widgets.Button(description="Run Tasks 3, 4 & 5 (Geom & Energy)", button_style='info', layout=widgets.Layout(width='auto'))
btn_task67 = widgets.Button(description="Run Tasks 6 & 7 (Convergence)", button_style='warning', layout=widgets.Layout(width='auto'))
btn_task8 = widgets.Button(description="Run Task 8 (Methods)", button_style='danger', layout=widgets.Layout(width='auto'))

# Thermo inputs for Task 10
thermo_corr = widgets.FloatText(value=0.0, description='ΔG_corr (eV):', tooltip='Input NIST thermodynamic corrections (ZPE, H, -TS) in eV')
btn_task10 = widgets.Button(description="Run Task 10 (Thermodynamics)", button_style='success', layout=widgets.Layout(width='auto'))

# --- 3. Execution Logic ---
def execute_tasks_345(b):
    with out:
        clear_output()
        print("--- Running Tasks 3, 4 & 5: Geometry Optimization & SCF Energies ---")
        basis = basis_dropdown.value
        
        # Build
        m_h2 = build_mol(h2_coords.value, basis, spin=0)
        m_o2 = build_mol(o2_coords.value, basis, spin=2)
        m_h2o = build_mol(h2o_coords.value, basis, spin=0)
        
        # Optimize
        print("Optimizing geometries...")
        m_h2_opt = optimize_quietly(scf.RHF(m_h2))
        m_o2_opt = optimize_quietly(scf.UHF(m_o2))
        m_h2o_opt = optimize_quietly(scf.RHF(m_h2o))
        
        # Structural Data (Task 3)
        print("\n--- Task 3: Geometry & Structural Data ---")
        print(f"H-H Bond Length : {get_bond_length(m_h2_opt, 0, 1):.4f} Å")
        print(f"O-O Bond Length : {get_bond_length(m_o2_opt, 0, 1):.4f} Å")
        print(f"H2O O-H1 Length : {get_bond_length(m_h2o_opt, 0, 1):.4f} Å")
        print(f"H2O O-H2 Length : {get_bond_length(m_h2o_opt, 0, 2):.4f} Å")
        print(f"H2O H-O-H Angle : {get_bond_angle(m_h2o_opt, 1, 0, 2):.2f}°")
        
        # Energies (Tasks 4 & 5)
        print("\n--- Tasks 4 & 5: Electronic Energy Data ---")
        e_h2 = scf.RHF(m_h2_opt).kernel()
        e_o2 = scf.UHF(m_o2_opt).kernel()
        e_h2o = scf.RHF(m_h2o_opt).kernel()
        
        e_r_hartree = (2 * e_h2 + e_o2) - (2 * e_h2o)
        e_r_ev = e_r_hartree * 27.2114
        
        print(f"E_tot (H2)  : {e_h2:.6f} Hartree")
        print(f"E_tot (O2)  : {e_o2:.6f} Hartree")
        print(f"E_tot (H2O) : {e_h2o:.6f} Hartree")
        print(f"Reaction Energy (ΔE_r): {e_r_hartree:.6f} Hartree | {e_r_ev:.4f} eV")

def execute_tasks_67(b):
    with out:
        clear_output()
        print("--- Running Tasks 6 & 7: Basis Set Convergence ---")
        basis_series = ['sto-3g', '3-21g', '6-31g', 'cc-pvdz'] 
        print(f"{'Basis Set':<12} | {'E_tot H2O (Hartree)':<20} | {'ΔE_r (eV)':<10}")
        print("-" * 50)
        
        for b_set in basis_series:
            m_h2 = build_mol(h2_coords.value, b_set, spin=0)
            m_o2 = build_mol(o2_coords.value, b_set, spin=2)
            m_h2o = build_mol(h2o_coords.value, b_set, spin=0)
            
            e_h2 = scf.RHF(m_h2).kernel()
            e_o2 = scf.UHF(m_o2).kernel()
            e_h2o = scf.RHF(m_h2o).kernel()
            
            e_r_ev = ((2 * e_h2 + e_o2) - (2 * e_h2o)) * 27.2114
            print(f"{b_set:<12} | {e_h2o:<20.6f} | {e_r_ev:<10.4f}")

def execute_task_8(b):
    with out:
        clear_output()
        print("--- Running Task 8: Method Comparison ---")
        basis = basis_dropdown.value
        m_h2 = build_mol(h2_coords.value, basis, spin=0)
        m_o2 = build_mol(o2_coords.value, basis, spin=2)
        m_h2o = build_mol(h2o_coords.value, basis, spin=0)
        
        methods = ["HF", "MP2", "CCSD", "DFT-PBE", "DFT-PBE0"]
        print(f"{'Method':<12} | {'ΔE_r (eV)':<10}")
        print("-" * 25)
        
        # HF Baseline
        mf_h2 = scf.RHF(m_h2).run()
        mf_o2 = scf.UHF(m_o2).run()
        mf_h2o = scf.RHF(m_h2o).run()
        
        for mod in methods:
            try:
                if mod == "HF":
                    eh2, eo2, eh2o = mf_h2.e_tot, mf_o2.e_tot, mf_h2o.e_tot
                elif mod == "MP2":
                    eh2 = mp.MP2(mf_h2).run().e_tot
                    eo2 = mp.UMP2(mf_o2).run().e_tot
                    eh2o = mp.MP2(mf_h2o).run().e_tot
                elif mod == "CCSD":
                    eh2 = cc.CCSD(mf_h2).run().e_tot
                    eo2 = cc.UCCSD(mf_o2).run().e_tot
                    eh2o = cc.CCSD(mf_h2o).run().e_tot
                elif mod == "DFT-PBE":
                    eh2 = run_calculation(m_h2, "DFT", "pbe").e_tot
                    eo2 = run_calculation(m_o2, "DFT", "pbe").e_tot
                    eh2o = run_calculation(m_h2o, "DFT", "pbe").e_tot
                elif mod == "DFT-PBE0":
                    eh2 = run_calculation(m_h2, "DFT", "pbe0").e_tot
                    eo2 = run_calculation(m_o2, "DFT", "pbe0").e_tot
                    eh2o = run_calculation(m_h2o, "DFT", "pbe0").e_tot

                er_ev = ((2 * eh2 + eo2) - (2 * eh2o)) * 27.2114
                print(f"{mod:<12} | {er_ev:<10.4f}")
            except Exception as e:
                print(f"{mod:<12} | Failed: {e}")

def execute_task_10(b):
    with out:
        clear_output()
        print("--- Running Task 10: Final Accuracy Data ---")
        basis = basis_dropdown.value
        m_h2 = build_mol(h2_coords.value, basis, spin=0)
        m_o2 = build_mol(o2_coords.value, basis, spin=2)
        m_h2o = build_mol(h2o_coords.value, basis, spin=0)
        
        # Getting basic SCF energy for the equation
        e_h2 = scf.RHF(m_h2).kernel()
        e_o2 = scf.UHF(m_o2).kernel()
        e_h2o = scf.RHF(m_h2o).kernel()
        
        e_r_ev = ((2 * e_h2 + e_o2) - (2 * e_h2o)) * 27.2114
        
        # Apply Thermodynamics
        g_corr = thermo_corr.value
        delta_g = e_r_ev + g_corr
        reference_g = 4.92
        deviation = delta_g - reference_g
        
        print(f"Calculated ΔE_r           : {e_r_ev:.4f} eV")
        print(f"Thermodynamic Correction  : {g_corr:.4f} eV")
        print(f"Final Calculated ΔG_r     : {delta_g:.4f} eV")
        print("-" * 40)
        print(f"Experimental Reference    : {reference_g:.4f} eV")
        print(f"Absolute Deviation        : {abs(deviation):.4f} eV")

# --- 4. Event Binding & Display ---
btn_task345.on_click(execute_tasks_345)
btn_task67.on_click(execute_tasks_67)
btn_task8.on_click(execute_task_8)
btn_task10.on_click(execute_task_10)

ui = widgets.VBox([
    widgets.HTML("<h2>Electrocatalysis Project Dashboard</h2>"),
    widgets.HBox([basis_dropdown]),
    widgets.HBox([h2_coords, o2_coords, h2o_coords]),
    widgets.HTML("<hr>"),
    widgets.HBox([btn_task345, btn_task67, btn_task8]),
    widgets.HBox([thermo_corr, btn_task10]),
    widgets.HTML("<hr>"),
    out
])

display(ui)

In [1]:
"""
Computational Electrocatalysis: Water Splitting Project (with plots)
"""

import numpy as np
import ipywidgets as widgets
from IPython.display import display, clear_output
import matplotlib.pyplot as plt
from pyscf import gto, scf, mp, cc, dft
import contextlib
import io
import logging

try:
    from pyscf.geomopt.geometric_solver import optimize
except ImportError:
    print("Warning: geomeTRIC not found. Geometry optimization will be skipped.")
    optimize = lambda mf: mf.mol # Fallback if geomeTRIC is missing

def optimize_quietly(mf):
    """Runs PySCF geometry optimization while completely suppressing geomeTRIC logs."""
    f = io.StringIO()
    with contextlib.redirect_stdout(f), contextlib.redirect_stderr(f):
        logger = logging.getLogger()
        old_level = logger.level
        logger.setLevel(logging.CRITICAL)
        try:
            opt_mol = optimize(mf)
        finally:
            logger.setLevel(old_level)
    return opt_mol

# --- 1. Helper Functions & Global State ---
BOHR_TO_ANGSTROM = 0.529177210903
EV_PER_HARTREE = 27.2114
TEMPERATURE = 298.15 # K

# Global dictionary to store optimized geometries between tasks
optimized_geometries = {
    'h2': None,
    'o2': None,
    'h2o': None
}

def build_mol(coords, basis, spin=0):
    return gto.M(atom=coords, basis=basis, spin=spin, symmetry=False, verbose=0)

def get_geom_string(mol):
    """Extracts PySCF optimized coordinates and formats them back to an atom string."""
    coords = mol.atom_coords() * BOHR_TO_ANGSTROM
    elements = [mol.atom_symbol(i) for i in range(mol.natm)]
    return "; ".join([f"{e} {c[0]:.6f} {c[1]:.6f} {c[2]:.6f}" for e, c in zip(elements, coords)])

def get_bond_length(mol, idx1, idx2):
    coords = mol.atom_coords() * BOHR_TO_ANGSTROM
    return np.linalg.norm(coords[idx1] - coords[idx2])

def get_bond_angle(mol, idx_a, idx_vertex, idx_c):
    coords = mol.atom_coords() * BOHR_TO_ANGSTROM
    v1 = coords[idx_a] - coords[idx_vertex]
    v2 = coords[idx_c] - coords[idx_vertex]
    cosine_angle = np.dot(v1, v2) / (np.linalg.norm(v1) * np.linalg.norm(v2))
    return np.degrees(np.arccos(np.clip(cosine_angle, -1.0, 1.0)))

def run_calculation(mol, method="HF", xc="pbe"):
    if method == "HF":
        mf = scf.UHF(mol) if mol.spin > 0 else scf.RHF(mol)
    elif method == "DFT":
        mf = dft.UKS(mol) if mol.spin > 0 else dft.RKS(mol)
        mf.xc = xc
    mf.kernel()
    return mf

# --- 2. Widget Definitions & UI Layout ---
out = widgets.Output()

basis_dropdown = widgets.Dropdown(options=['sto-3g', '3-21g', '6-31g', 'cc-pvdz', 'cc-pvtz'], value='sto-3g', description='Base Basis:')
h2_coords = widgets.Text(value='H 0 0 0; H 0 0 0.74', description='H2 Coords:')
o2_coords = widgets.Text(value='O 0 0 0; O 0 0 1.21', description='O2 Coords:')
h2o_coords = widgets.Text(value='O 0 0 0; H 0 0.757 0.586; H 0 -0.757 0.586', description='H2O Coords:')

btn_task345 = widgets.Button(description="Run Tasks 3-5 (Geom & Energy)", button_style='info', layout=widgets.Layout(width='auto'))
btn_task67 = widgets.Button(description="Run Tasks 6 & 7 (Convergence Plots)", button_style='warning', layout=widgets.Layout(width='auto'))
btn_task8 = widgets.Button(description="Run Task 8 (Methods)", button_style='danger', layout=widgets.Layout(width='auto'))
btn_task10 = widgets.Button(description="Run Task 10 (Thermodynamics)", button_style='success', layout=widgets.Layout(width='auto'))

# --- 3. Execution Logic ---
def execute_tasks_345(b):
    with out:
        clear_output()
        print("--- Running Tasks 3, 4 & 5: Geometry Optimization & SCF Energies ---")
        basis = basis_dropdown.value
        
        m_h2 = build_mol(h2_coords.value, basis, spin=0)
        m_o2 = build_mol(o2_coords.value, basis, spin=2)
        m_h2o = build_mol(h2o_coords.value, basis, spin=0)
        
        print("Optimizing geometries at HF/STO-3G level...")
        m_h2_opt = optimize_quietly(scf.RHF(m_h2))
        m_o2_opt = optimize_quietly(scf.UHF(m_o2))
        m_h2o_opt = optimize_quietly(scf.RHF(m_h2o))
        
        # Save optimized geometries globally for subsequent tasks
        global optimized_geometries
        optimized_geometries['h2'] = get_geom_string(m_h2_opt)
        optimized_geometries['o2'] = get_geom_string(m_o2_opt)
        optimized_geometries['h2o'] = get_geom_string(m_h2o_opt)
        
        print("\n--- Task 3: Geometry & Structural Data ---")
        print(f"H-H Bond Length : {get_bond_length(m_h2_opt, 0, 1):.4f} Å")
        print(f"O-O Bond Length : {get_bond_length(m_o2_opt, 0, 1):.4f} Å")
        print(f"H2O O-H1 Length : {get_bond_length(m_h2o_opt, 0, 1):.4f} Å")
        print(f"H2O H-O-H Angle : {get_bond_angle(m_h2o_opt, 1, 0, 2):.2f}°")
        
        print("\n--- Tasks 4 & 5: Electronic Energy Data ---")
        e_h2 = scf.RHF(m_h2_opt).kernel()
        e_o2 = scf.UHF(m_o2_opt).kernel()
        e_h2o = scf.RHF(m_h2o_opt).kernel()
        
        e_r_hartree = (2 * e_h2 + e_o2) - (2 * e_h2o)
        e_r_ev = e_r_hartree * EV_PER_HARTREE
        
        print(f"E_tot (H2)  : {e_h2:.6f} Hartree")
        print(f"E_tot (O2)  : {e_o2:.6f} Hartree")
        print(f"E_tot (H2O) : {e_h2o:.6f} Hartree")
        print(f"Reaction Energy (ΔE_r): {e_r_hartree:.6f} Hartree | {e_r_ev:.4f} eV")


def execute_tasks_67(b):
    with out:
        clear_output()
        if optimized_geometries['h2'] is None:
            print("Please run Tasks 3-5 first to optimize the geometries!")
            return
            
        print("--- Running Tasks 6 & 7: Basis Set Convergence ---")
        print("Using FIXED geometries from Task 3.")
        
        # Note: cc-pV5Z is excluded by default in live widgets to prevent crashing. 
        # Add 'cc-pv5z' to this list if your environment can handle the calculation time.
        basis_series = ['sto-3g', '3-21g', '6-31g', 'cc-pvdz', 'cc-pvtz', 'cc-pvqz'] 
        
        e_reaction_list = []
        e_h2o_list = []
        
        for b_set in basis_series:
            print(f"Running HF/{b_set}...")
            m_h2 = build_mol(optimized_geometries['h2'], b_set, spin=0)
            m_o2 = build_mol(optimized_geometries['o2'], b_set, spin=2)
            m_h2o = build_mol(optimized_geometries['h2o'], b_set, spin=0)
            
            e_h2 = scf.RHF(m_h2).kernel()
            e_o2 = scf.UHF(m_o2).kernel()
            e_h2o = scf.RHF(m_h2o).kernel()
            
            e_r_ev = ((2 * e_h2 + e_o2) - (2 * e_h2o)) * EV_PER_HARTREE
            
            e_reaction_list.append(e_r_ev)
            e_h2o_list.append(e_h2o)
            
        # Matplotlib visualization for Tasks 6 & 7
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
        
        # Task 6: Reaction Energy Plot
        ax1.plot(basis_series, e_reaction_list, marker='o', color='b', linestyle='-')
        ax1.set_title("Task 6: Reaction Energy vs Basis Set")
        ax1.set_ylabel("ΔE_r (eV)")
        ax1.set_xlabel("Basis Set")
        ax1.grid(True)
        
        # Task 7: H2O Energy vs Reaction Energy Convergence
        ax2.plot(basis_series, e_h2o_list, marker='s', color='r', label='E_tot H2O (Hartree)')
        ax2.set_title("Task 7: H2O Molecule Energy Convergence")
        ax2.set_ylabel("Total Energy (Hartree)")
        ax2.set_xlabel("Basis Set")
        ax2.grid(True)
        
        plt.tight_layout()
        plt.show()


def execute_task_8(b):
    with out:
        clear_output()
        if optimized_geometries['h2'] is None:
            print("Please run Tasks 3-5 first to optimize the geometries!")
            return
            
        print("--- Running Task 8: Method Comparison ---")
        print("Using fixed STO-3G geometries. Basis set: cc-pVDZ (for time efficiency in demo)")
        
        demo_basis = 'cc-pvdz' 
        m_h2 = build_mol(optimized_geometries['h2'], demo_basis, spin=0)
        m_o2 = build_mol(optimized_geometries['o2'], demo_basis, spin=2)
        m_h2o = build_mol(optimized_geometries['h2o'], demo_basis, spin=0)
        
        methods = ["HF", "MP2", "CCSD", "DFT-PBE", "DFT-PBE0"]
        print(f"\n{'Method':<12} | {'ΔE_r (eV)':<10}")
        print("-" * 25)
        
        mf_h2 = scf.RHF(m_h2).run()
        mf_o2 = scf.UHF(m_o2).run()
        mf_h2o = scf.RHF(m_h2o).run()
        
        for mod in methods:
            try:
                if mod == "HF":
                    eh2, eo2, eh2o = mf_h2.e_tot, mf_o2.e_tot, mf_h2o.e_tot
                elif mod == "MP2":
                    eh2 = mp.MP2(mf_h2).run().e_tot
                    eo2 = mp.UMP2(mf_o2).run().e_tot
                    eh2o = mp.MP2(mf_h2o).run().e_tot
                elif mod == "CCSD":
                    eh2 = cc.CCSD(mf_h2).run().e_tot
                    eo2 = cc.UCCSD(mf_o2).run().e_tot
                    eh2o = cc.CCSD(mf_h2o).run().e_tot
                elif mod == "DFT-PBE":
                    eh2 = run_calculation(m_h2, "DFT", "pbe").e_tot
                    eo2 = run_calculation(m_o2, "DFT", "pbe").e_tot
                    eh2o = run_calculation(m_h2o, "DFT", "pbe").e_tot
                elif mod == "DFT-PBE0":
                    eh2 = run_calculation(m_h2, "DFT", "pbe0").e_tot
                    eo2 = run_calculation(m_o2, "DFT", "pbe0").e_tot
                    eh2o = run_calculation(m_h2o, "DFT", "pbe0").e_tot

                er_ev = ((2 * eh2 + eo2) - (2 * eh2o)) * EV_PER_HARTREE
                print(f"{mod:<12} | {er_ev:<10.4f}")
            except Exception as e:
                print(f"{mod:<12} | Failed: {e}")


def execute_task_10(b):
    with out:
        clear_output()
        print("--- Running Task 10: Thermodynamic Accuracy ---")
        print("Re-relaxing geometries at DFT-PBE/6-31G to calculate exact Gibbs Free Energy...\n")
        
        # Re-relaxation step required by Task 10
        m_h2 = build_mol(h2_coords.value, '6-31g', spin=0)
        m_o2 = build_mol(o2_coords.value, '6-31g', spin=2)
        m_h2o = build_mol(h2o_coords.value, '6-31g', spin=0)
        
        mf_h2 = dft.RKS(m_h2); mf_h2.xc = 'pbe'
        mf_o2 = dft.UKS(m_o2); mf_o2.xc = 'pbe'
        mf_h2o = dft.RKS(m_h2o); mf_h2o.xc = 'pbe'
        
        m_h2_opt = optimize_quietly(mf_h2)
        m_o2_opt = optimize_quietly(mf_o2)
        m_h2o_opt = optimize_quietly(mf_h2o)
        
        e_h2 = dft.RKS(m_h2_opt).kernel()
        e_o2 = dft.UKS(m_o2_opt).kernel()
        e_h2o = dft.RKS(m_h2o_opt).kernel()
        
        # --- PLACEHOLDER THERMO DATA --- 
        # You MUST replace these placeholder values with actual NIST data for your report.
        # Units assumed here: ZPE (eV), H (eV), S (eV/K)
        thermo_data = {
            'H2_g':  {'ZPE': 0.273, 'H_corr': 0.089, 'S': 0.0013},
            'O2_g':  {'ZPE': 0.098, 'H_corr': 0.089, 'S': 0.0021},
            'H2O_g': {'ZPE': 0.573, 'H_corr': 0.103, 'S': 0.0019}
        }
        
        # H2O Liquid correction phase changes (H_liq - H_gas) and (S_liq - S_gas)
        h2o_liq_H_diff = -0.456  # eV
        h2o_liq_S_diff = -0.0012 # eV/K

        def calc_G_gas(E_elec, data):
            return (E_elec * EV_PER_HARTREE) + data['ZPE'] + data['H_corr'] - (TEMPERATURE * data['S'])
            
        G_H2_g = calc_G_gas(e_h2, thermo_data['H2_g'])
        G_O2_g = calc_G_gas(e_o2, thermo_data['O2_g'])
        G_H2O_g = calc_G_gas(e_h2o, thermo_data['H2O_g'])
        
        # Liquid water thermodynamic cycle correction
        G_H2O_l = G_H2O_g + h2o_liq_H_diff - (TEMPERATURE * h2o_liq_S_diff)
        
        # Final reaction free energy
        delta_G_r = (2 * G_H2_g + G_O2_g) - (2 * G_H2O_l)
        reference_G = 4.92 
        
        print(f"Calculated ΔG_r (DFT-PBE) : {delta_G_r:.4f} eV")
        print(f"Experimental Reference    : {reference_G:.4f} eV")
        print(f"Absolute Error            : {abs(delta_G_r - reference_G):.4f} eV")
        
        # Generate Bar Chart
        fig, ax = plt.subplots(figsize=(6,4))
        categories = ['DFT-PBE Calculated', 'Literature Value']
        values = [delta_G_r, reference_G]
        
        ax.bar(categories, values, color=['#1f77b4', '#2ca02c'])
        ax.set_ylabel("ΔG_r (eV)")
        ax.set_title("Task 10: Accuracy of Thermodynamically Corrected Reaction Energy")
        ax.axhline(reference_G, color='r', linestyle='--', label='Lit. Reference')
        ax.legend()
        
        plt.tight_layout()
        plt.show()

# --- 4. Event Binding & Display ---
btn_task345.on_click(execute_tasks_345)
btn_task67.on_click(execute_tasks_67)
btn_task8.on_click(execute_task_8)
btn_task10.on_click(execute_task_10)

ui = widgets.VBox([
    widgets.HTML("<h2>Electrocatalysis Project Dashboard</h2>"),
    widgets.HTML("<p><i>Ensure you have run Tasks 3-5 to set the optimized geometries before running the latter tasks.</i></p>"),
    widgets.HBox([basis_dropdown]),
    widgets.HBox([h2_coords, o2_coords, h2o_coords]),
    widgets.HTML("<hr>"),
    widgets.HBox([btn_task345, btn_task67]),
    widgets.HBox([btn_task8, btn_task10]),
    widgets.HTML("<hr>"),
    out
])

display(ui)

In [ ]:
"""
Computational Electrocatalysis: Water Splitting Project (Fully Compliant) 
"""

import numpy as np
import ipywidgets as widgets
from IPython.display import display, clear_output
import matplotlib.pyplot as plt
from pyscf import gto, scf, mp, cc, dft
import contextlib
import io
import logging

try:
    from pyscf.geomopt.geometric_solver import optimize
except ImportError:
    print("Warning: geomeTRIC not found. Geometry optimization will be skipped.")
    optimize = lambda mf: mf.mol

def optimize_quietly(mf):
    f = io.StringIO()
    with contextlib.redirect_stdout(f), contextlib.redirect_stderr(f):
        logger = logging.getLogger()
        old_level = logger.level
        logger.setLevel(logging.CRITICAL)
        try:
            opt_mol = optimize(mf)
        finally:
            logger.setLevel(old_level)
    return opt_mol

# --- 1. Helper Functions & Global State ---
BOHR_TO_ANGSTROM = 0.529177210903
EV_PER_HARTREE = 27.2114
TEMPERATURE = 298.15 # K
LIT_REF_G = 4.92 # eV

optimized_geometries = {'h2': None, 'o2': None, 'h2o': None}

def build_mol(coords, basis, spin=0):
    return gto.M(atom=coords, basis=basis, spin=spin, symmetry=False, verbose=0)

def get_geom_string(mol):
    coords = mol.atom_coords() * BOHR_TO_ANGSTROM
    elements = [mol.atom_symbol(i) for i in range(mol.natm)]
    return "; ".join([f"{e} {c[0]:.6f} {c[1]:.6f} {c[2]:.6f}" for e, c in zip(elements, coords)])

def get_bond_length(mol, idx1, idx2):
    coords = mol.atom_coords() * BOHR_TO_ANGSTROM
    return np.linalg.norm(coords[idx1] - coords[idx2])

def get_bond_angle(mol, idx_a, idx_vertex, idx_c):
    coords = mol.atom_coords() * BOHR_TO_ANGSTROM
    v1 = coords[idx_a] - coords[idx_vertex]
    v2 = coords[idx_c] - coords[idx_vertex]
    cosine_angle = np.dot(v1, v2) / (np.linalg.norm(v1) * np.linalg.norm(v2))
    return np.degrees(np.arccos(np.clip(cosine_angle, -1.0, 1.0)))

def run_calculation(mol, method="HF", xc="pbe"):
    if method == "HF":
        mf = scf.UHF(mol) if mol.spin > 0 else scf.RHF(mol)
    elif method == "DFT":
        mf = dft.UKS(mol) if mol.spin > 0 else dft.RKS(mol)
        mf.xc = xc
    mf.kernel()
    return mf

# --- 2. Widget Definitions ---
out = widgets.Output()

basis_dropdown = widgets.Dropdown(
    options=['sto-3g', '3-21g', '6-31g', 'cc-pvdz', 'cc-pvtz', 'cc-pvqz'], 
    value='cc-pvdz', 
    description='Target Basis:'
)
h2_coords = widgets.Text(value='H 0 0 0; H 0 0 0.74', description='H2 Coords:')
o2_coords = widgets.Text(value='O 0 0 0; O 0 0 1.21', description='O2 Coords:')
h2o_coords = widgets.Text(value='O 0 0 0; H 0 0.757 0.586; H 0 -0.757 0.586', description='H2O Coords:')

btn_task345 = widgets.Button(description="Run Tasks 3-5 (Geom & Energy)", button_style='info', layout=widgets.Layout(width='auto'))
btn_task67 = widgets.Button(description="Run Tasks 6 & 7 (Convergence Plots)", button_style='warning', layout=widgets.Layout(width='auto'))
btn_task8 = widgets.Button(description="Run Task 8 (Method Ladder)", button_style='danger', layout=widgets.Layout(width='auto'))
btn_task10 = widgets.Button(description="Run Task 10 (Full Thermo & Bar Chart)", button_style='success', layout=widgets.Layout(width='auto'))

# --- 3. Execution Logic ---
def execute_tasks_345(b):
    with out:
        clear_output()
        print("--- Running Tasks 3, 4 & 5: Geometry Optimization & SCF Energies ---")
        basis = 'sto-3g' # T2 requires HF/STO-3G optimization
        
        m_h2 = build_mol(h2_coords.value, basis, spin=0)
        m_o2 = build_mol(o2_coords.value, basis, spin=2)
        m_h2o = build_mol(h2o_coords.value, basis, spin=0)
        
        print("Optimizing geometries at HF/STO-3G level...")
        m_h2_opt = optimize_quietly(scf.RHF(m_h2))
        m_o2_opt = optimize_quietly(scf.UHF(m_o2))
        m_h2o_opt = optimize_quietly(scf.RHF(m_h2o))
        
        global optimized_geometries
        optimized_geometries['h2'] = get_geom_string(m_h2_opt)
        optimized_geometries['o2'] = get_geom_string(m_o2_opt)
        optimized_geometries['h2o'] = get_geom_string(m_h2o_opt)
        
        print("\n--- Task 3: Geometry Analysis ---")
        print(f"H-H Bond Length : {get_bond_length(m_h2_opt, 0, 1):.4f} Å")
        print(f"O-O Bond Length : {get_bond_length(m_o2_opt, 0, 1):.4f} Å")
        print(f"H2O O-H1 Length : {get_bond_length(m_h2o_opt, 0, 1):.4f} Å")
        print(f"H2O H-O-H Angle : {get_bond_angle(m_h2o_opt, 1, 0, 2):.2f}°")
        
        print("\n--- Tasks 4 & 5: Electronic Energies ---")
        e_h2 = scf.RHF(m_h2_opt).kernel()
        e_o2 = scf.UHF(m_o2_opt).kernel()
        e_h2o = scf.RHF(m_h2o_opt).kernel()
        
        e_r_hartree = (2 * e_h2 + e_o2) - (2 * e_h2o)
        e_r_ev = e_r_hartree * EV_PER_HARTREE
        
        print(f"E_tot (H2)  : {e_h2:.6f} Hartree")
        print(f"E_tot (O2)  : {e_o2:.6f} Hartree")
        print(f"E_tot (H2O) : {e_h2o:.6f} Hartree")
        print(f"Reaction Energy (ΔE_r): {e_r_hartree:.6f} Hartree | {e_r_ev:.4f} eV")

def execute_tasks_67(b):
    with out:
        clear_output()
        if optimized_geometries['h2'] is None:
            print("Error: Run Tasks 3-5 first to generate HF/STO-3G optimized geometries!")
            return
            
        print("--- Tasks 6 & 7: Basis Set Convergence Study ---")
        basis_series = ['sto-3g', '3-21g', '6-31g', 'cc-pvdz', 'cc-pvtz', 'cc-pvqz']
        
        e_reaction_list = []
        e_h2o_list = []
        
        for b_set in basis_series:
            print(f"Calculating HF/{b_set}...")
            m_h2 = build_mol(optimized_geometries['h2'], b_set, spin=0)
            m_o2 = build_mol(optimized_geometries['o2'], b_set, spin=2)
            m_h2o = build_mol(optimized_geometries['h2o'], b_set, spin=0)
            
            e_h2 = scf.RHF(m_h2).kernel()
            e_o2 = scf.UHF(m_o2).kernel()
            e_h2o = scf.RHF(m_h2o).kernel()
            
            e_r_ev = ((2 * e_h2 + e_o2) - (2 * e_h2o)) * EV_PER_HARTREE
            
            e_reaction_list.append(e_r_ev)
            e_h2o_list.append(e_h2o)
            
        # Quantitative comparison vs cc-pVQZ reference (Task 6)
        ref_e_r = e_reaction_list[-1] # cc-pVQZ result
        print(f"\n{'Basis Set':<12} | {'ΔE_r (eV)':<12} | {'Error vs cc-pVQZ (eV)':<20}")
        print("-" * 50)
        for b_set, er in zip(basis_series, e_reaction_list):
            diff = er - ref_e_r
            print(f"{b_set:<12} | {er:<12.4f} | {diff:<20.4f}")

        # Plotting Task 6 & Task 7
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
        
        # Plot Task 6
        ax1.plot(basis_series, e_reaction_list, marker='o', color='b', linestyle='-')
        ax1.axhline(ref_e_r, color='r', linestyle='--', label='cc-pVQZ Ref')
        ax1.set_title("Task 6: Reaction Energy Convergence")
        ax1.set_ylabel("ΔE_r (eV)")
        ax1.set_xlabel("Basis Set")
        ax1.grid(True)
        ax1.legend()
        
        # Plot Task 7: Relative convergence comparison (Error relative to largest basis set in eV)
        h2o_rel_ev = [(e - e_h2o_list[-1]) * EV_PER_HARTREE for e in e_h2o_list]
        rxn_rel_ev = [e - ref_e_r for e in e_reaction_list]
        
        ax2.plot(basis_series, h2o_rel_ev, marker='s', color='red', label='H2O Single Mol Error (eV)')
        ax2.plot(basis_series, rxn_rel_ev, marker='o', color='blue', label='Reaction ΔE_r Error (eV)')
        ax2.set_title("Task 7: Single Molecule vs Reaction Energy Convergence")
        ax2.set_ylabel("Energy Error relative to cc-pVQZ (eV)")
        ax2.set_xlabel("Basis Set")
        ax2.grid(True)
        ax2.legend()
        
        plt.tight_layout()
        plt.show()

def execute_task_8(b):
    with out:
        clear_output()
        if optimized_geometries['h2'] is None:
            print("Error: Run Tasks 3-5 first!")
            return
            
        selected_basis = basis_dropdown.value
        print(f"--- Task 8: Method Ladder Comparison (Basis: {selected_basis}) ---")
        
        m_h2 = build_mol(optimized_geometries['h2'], selected_basis, spin=0)
        m_o2 = build_mol(optimized_geometries['o2'], selected_basis, spin=2)
        m_h2o = build_mol(optimized_geometries['h2o'], selected_basis, spin=0)
        
        methods = ["HF", "MP2", "CCSD", "DFT-PBE", "DFT-PBE0"]
        print(f"\n{'Method':<12} | {'ΔE_r (eV)':<10}")
        print("-" * 25)
        
        mf_h2 = scf.RHF(m_h2).run()
        mf_o2 = scf.UHF(m_o2).run()
        mf_h2o = scf.RHF(m_h2o).run()
        
        for mod in methods:
            try:
                if mod == "HF":
                    eh2, eo2, eh2o = mf_h2.e_tot, mf_o2.e_tot, mf_h2o.e_tot
                elif mod == "MP2":
                    eh2 = mp.MP2(mf_h2).run().e_tot
                    eo2 = mp.UMP2(mf_o2).run().e_tot
                    eh2o = mp.MP2(mf_h2o).run().e_tot
                elif mod == "CCSD":
                    eh2 = cc.CCSD(mf_h2).run().e_tot
                    eo2 = cc.UCCSD(mf_o2).run().e_tot
                    eh2o = cc.CCSD(mf_h2o).run().e_tot
                elif mod == "DFT-PBE":
                    eh2 = run_calculation(m_h2, "DFT", "pbe").e_tot
                    eo2 = run_calculation(m_o2, "DFT", "pbe").e_tot
                    eh2o = run_calculation(m_h2o, "DFT", "pbe").e_tot
                elif mod == "DFT-PBE0":
                    eh2 = run_calculation(m_h2, "DFT", "pbe0").e_tot
                    eo2 = run_calculation(m_o2, "DFT", "pbe0").e_tot
                    eh2o = run_calculation(m_h2o, "DFT", "pbe0").e_tot

                er_ev = ((2 * eh2 + eo2) - (2 * eh2o)) * EV_PER_HARTREE
                print(f"{mod:<12} | {er_ev:<10.4f}")
            except Exception as e:
                print(f"{mod:<12} | Failed: {e}")

def execute_task_10(b):
    with out:
        clear_output()
        selected_basis = basis_dropdown.value
        print(f"--- Task 10: Thermodynamic Corrections & Method Deviation Analysis ---")
        print(f"Basis Set: {selected_basis} | Temperature: 298.15 K\n")
        
        # NIST Thermodynamic Data (Replace placeholders with actual NIST CCCBDB values for report)
        thermo_data = {
            'H2_g':  {'ZPE': 0.273, 'H_corr': 0.089, 'S': 0.0013},
            'O2_g':  {'ZPE': 0.098, 'H_corr': 0.089, 'S': 0.0021},
            'H2O_g': {'ZPE': 0.573, 'H_corr': 0.103, 'S': 0.0019}
        }
        h2o_liq_H_diff = -0.456  # eV
        h2o_liq_S_diff = -0.0012 # eV/K

        def calc_G_gas(E_elec, data):
            return (E_elec * EV_PER_HARTREE) + data['ZPE'] + data['H_corr'] - (TEMPERATURE * data['S'])

        methods = ["HF", "MP2", "CCSD", "DFT-PBE", "DFT-PBE0"]
        delta_G_results = {}
        deviations = {}

        for mod in methods:
            print(f"Re-relaxing and calculating thermodynamics for {mod}...")
            m_h2 = build_mol(h2_coords.value, selected_basis, spin=0)
            m_o2 = build_mol(o2_coords.value, selected_basis, spin=2)
            m_h2o = build_mol(h2o_coords.value, selected_basis, spin=0)
            
            # Re-relax geometry for each method
            if mod in ["DFT-PBE", "DFT-PBE0"]:
                xc_name = "pbe" if mod == "DFT-PBE" else "pbe0"
                mf_h2 = dft.RKS(m_h2); mf_h2.xc = xc_name
                mf_o2 = dft.UKS(m_o2); mf_o2.xc = xc_name
                mf_h2o = dft.RKS(m_h2o); mf_h2o.xc = xc_name
            else:
                mf_h2 = scf.RHF(m_h2)
                mf_o2 = scf.UHF(m_o2)
                mf_h2o = scf.RHF(m_h2o)

            m_h2_opt = optimize_quietly(mf_h2)
            m_o2_opt = optimize_quietly(mf_o2)
            m_h2o_opt = optimize_quietly(mf_h2o)

            # Evaluate total electronic energy
            if mod == "HF":
                eh2 = scf.RHF(m_h2_opt).kernel()
                eo2 = scf.UHF(m_o2_opt).kernel()
                eh2o = scf.RHF(m_h2o_opt).kernel()
            elif mod == "MP2":
                eh2 = mp.MP2(scf.RHF(m_h2_opt).run()).run().e_tot
                eo2 = mp.UMP2(scf.UHF(m_o2_opt).run()).run().e_tot
                eh2o = mp.MP2(scf.RHF(m_h2o_opt).run()).run().e_tot
            elif mod == "CCSD":
                eh2 = cc.CCSD(scf.RHF(m_h2_opt).run()).run().e_tot
                eo2 = cc.UCCSD(scf.UHF(m_o2_opt).run()).run().e_tot
                eh2o = cc.CCSD(scf.RHF(m_h2o_opt).run()).run().e_tot
            elif mod in ["DFT-PBE", "DFT-PBE0"]:
                xc_name = "pbe" if mod == "DFT-PBE" else "pbe0"
                eh2 = run_calculation(m_h2_opt, "DFT", xc_name).e_tot
                eo2 = run_calculation(m_o2_opt, "DFT", xc_name).e_tot
                eh2o = run_calculation(m_h2o_opt, "DFT", xc_name).e_tot

            # Apply Gibbs Free Energy Formulas
            G_H2_g = calc_G_gas(eh2, thermo_data['H2_g'])
            G_O2_g = calc_G_gas(eo2, thermo_data['O2_g'])
            G_H2O_g = calc_G_gas(eh2o, thermo_data['H2O_g'])
            G_H2O_l = G_H2O_g + h2o_liq_H_diff - (TEMPERATURE * h2o_liq_S_diff)

            dG_r = (2 * G_H2_g + G_O2_g) - (2 * G_H2O_l)
            delta_G_results[mod] = dG_r
            deviations[mod] = abs(dG_r - LIT_REF_G)

        # Print Table of Results
        print(f"\n{'Method':<12} | {'Calculated ΔG_r (eV)':<20} | {'Absolute Error vs Lit (eV)':<25}")
        print("-" * 62)
        for mod in methods:
            print(f"{mod:<12} | {delta_G_results[mod]:<20.4f} | {deviations[mod]:<25.4f}")

        # Multi-Method Deviation Bar Chart
        fig, ax = plt.subplots(figsize=(8, 5))
        ax.bar(methods, [deviations[m] for m in methods], color='skyblue', edgecolor='black')
        ax.set_ylabel("Absolute Error |ΔG_calc - 4.92 eV| (eV)")
        ax.set_title("Task 10: Deviation of Thermodynamically Corrected Reaction Energy by Method")
        ax.grid(axis='y', linestyle='--', alpha=0.7)
        
        plt.tight_layout()
        plt.show()

# --- 4. Event Binding & Display ---
btn_task345.on_click(execute_tasks_345)
btn_task67.on_click(execute_tasks_67)
btn_task8.on_click(execute_task_8)
btn_task10.on_click(execute_task_10)

ui = widgets.VBox([
    widgets.HTML("<h2>Electrocatalysis Project Dashboard</h2>"),
    widgets.HBox([basis_dropdown]),
    widgets.HBox([h2_coords, o2_coords, h2o_coords]),
    widgets.HTML("<hr>"),
    widgets.HBox([btn_task345, btn_task67]),
    widgets.HBox([btn_task8, btn_task10]),
    widgets.HTML("<hr>"),
    out
])

display(ui)